In [1]:
# Use a pipeline as a high-level helper
from transformers import pipeline
'''import libraries used, json to read the data, 
randint for randomly generated paragraphs, 
and pandas to create a dataframe to be written to csv'''
import pandas as pd
from tqdm.notebook import tqdm
import json
import csv
#fname = 'dev-SQuAD-v2.0'
#f = open(fname + '.json')
#load in data as python dictionary using the json library
#data = json.load(f)
#data = data["data"]
#samples = pd.read_csv("Negation_dev.csv")
#originals = pd.read_csv("dev_start.csv")
#originals = originals.drop('New_Context', axis=1)
#originals.drop(originals.columns[originals.columns.str.contains(
#    'unnamed', case=False)], axis=1, inplace=True)
#albert = pipeline("question-answering", model="madlag/albert-base-v2-squad")
electra = pipeline("question-answering", model="deepset/electra-base-squad2", device = "cuda")
roberta = pipeline("question-answering", model="deepset/roberta-large-squad2", device = "cuda")
model2 = pipeline("question-answering", model="timpal0l/mdeberta-v3-base-squad2", device = "cuda")
model4 = pipeline("question-answering", model="AVISHKAARAM/avishkaarak-ekta-hindi", device = "cuda")
fname = "BARTEPOCH3"

2024-08-21 13:07:19.623109: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-21 13:07:19.661689: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-21 13:07:20.451733: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(

In [2]:
df = pd.read_csv("after_3_epochs.csv")
df

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,qid,context,original_question,label,answer span,original_answer_span,answer_sentence,BART_ANS,BART_LABEL
0,0,5,5,5ad39d53604f3c001a3fe8d1,The Normans (Norman: Nourmands; French: Norman...,Who gave their name to Normandy in the 1000's ...,I,"('', '')",Normans,The Normans (Norman: Nourmands; French: Norman...,Who gave their name to Normandy in the 10th an...,A
1,1,6,6,5ad39d53604f3c001a3fe8d2,The Normans (Norman: Nourmands; French: Norman...,What is France a region of?,I,"('', '')",Normandy,The Normans (Norman: Nourmands; French: Norman...,What is Normandy a region of?,A
2,2,7,7,5ad39d53604f3c001a3fe8d3,The Normans (Norman: Nourmands; French: Norman...,Who did King Charles III swear fealty to?,I,"('', '')",Rollo,"They were descended from Norse (""Norman"" comes...",Who did King Charles III swear fealty to?,A
3,3,8,8,5ad39d53604f3c001a3fe8d4,The Normans (Norman: Nourmands; French: Norman...,When did the Frankish identity emerge?,E,"('', '')",10th century,The distinct cultural and ethnic identity of t...,When did the Normans identity emerge?,A
4,4,12,12,5ad3a266604f3c001a3fea27,"The Norman dynasty had a major political, cult...",What type of major impact did the Norman dynas...,X,"('', '')","political, cultural and military","The Norman dynasty had a major political, cult...",What type of major impact did the Norman dynas...,E
...,...,...,...,...,...,...,...,...,...,...,...,...
5940,5940,11863,11863,5ad28a57d7d075001a4299b3,The connection between macroscopic nonconserva...,What does not change macroscopic closed systems?,N,"('', '')",nonconservative forces,The connection between macroscopic nonconserva...,What does change macroscopic closed systems?,A
5941,5941,11869,11869,5ad28ad0d7d075001a4299cc,"The pound-force has a metric counterpart, less...",What does not have a metric counterpart?,N,"('', '')",pound-force,"The pound-force has a metric counterpart, less...",What is the metric counterpart of the pound-fo...,I
5942,5942,11870,11870,5ad28ad0d7d075001a4299cd,"The pound-force has a metric counterpart, less...",What is the force exerted by standard gravity ...,E,"('', '')",kilogram-force,"The pound-force has a metric counterpart, less...",What is the pound-force primarily used for?,I
5943,5943,11871,11871,5ad28ad0d7d075001a4299ce,"The pound-force has a metric counterpart, less...",What force leads to a commonly used unit of mass?,E,"('', '')",kilogram-force,"The pound-force has a metric counterpart, less...",What is the metric counterpart of the pound-fo...,I


In [3]:
labels = ["model2", "model4", "electra", "roberta"]

In [4]:
final = []

In [5]:
#odata = json.load(open("SGO.json"))
#odata = [json.load(open("albert0.json")), json.load(open("SGO.json")), json.load(open("retrospective.json"))]
def get_output_models(id, oquestion, squestion, context):
    row = {}
    row["qid"] = id
    models = [model2, model4, electra, roberta]
    for i in range(0, len(models)):
        preds = models[i](question=oquestion, context=context,)
        #if(labels[i] != "albert"):
        row["OU " + labels[i] + " answer span"] = preds["answer"]
        row["OU " + labels[i] + " start"] = preds["start"]
        row["OU " + labels[i] + " end"] = preds["end"]
        row["OU " + labels[i] + " score"] = preds["score"]
        preds = models[i](question=squestion, context=context,)
        #if(labels[i] != "albert"):
        row["SA " + labels[i] + " answer span"] = preds["answer"]
        row["SA " + labels[i] + " start"] = preds["start"]
        row["SA " + labels[i] + " end"] = preds["end"]
        row["SA " + labels[i] + " score"] = preds["score"]
    #row["P-U " + labels[i + 1] + " answer span"] = odata[id] 
    #for j in range(0, len(odata)):
#        row["P-U " + labels[i + j] + " answer span"] = odata[j][id]
    return row

In [6]:
def save_csv(df):
    final_df = pd.DataFrame(final)
    final_df.to_csv(f"{fname}.csv")
    

In [7]:
#this is the dataframe that will be converted to pandas and then to csv
#final = []
#counter = 0
#loop through all different data titles/topics associated!
for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    if final == [] or row['qid'] not in pd.DataFrame(final)['qid'].values:
        sq = row["BART_ANS"]
        oq = row["original_question"]
        context = row["context"]
        try:
            frow = get_output_models(row['qid'], oq, str(sq), context)
            final.append(frow)
            save_csv(final)   
        except:
            pass
                    

            

  0%|          | 0/5945 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [8]:
final_df = pd.DataFrame(final)

In [9]:
final_df

,qid,OU model2 answer span,OU model2 start,OU model2 end,OU model2 score,SA model2 answer span,SA model2 start,SA model2 end,SA model2 score,OU model4 answer span,...,SA electra end,SA electra score,OU roberta answer span,OU roberta start,OU roberta end,OU roberta score,SA roberta answer span,SA roberta start,SA roberta end,SA roberta score
0,5ad39d53604f3c001a3fe8d1,The Normans,0,11,2.337817e-04,The Normans,0,11,8.394209e-01,The Normans,...,11,7.099302e-08,The Normans,0,11,4.186608e-06,The Normans,0,11,9.732896e-01
1,5ad39d53604f3c001a3fe8d2,"Normandy,",136,146,5.196336e-07,"Normandy,",136,146,5.196336e-07,Normandy,...,145,1.146701e-08,Normandy,137,145,4.917059e-08,Normandy,137,145,4.917059e-08
2,5ad39d53604f3c001a3fe8d3,West Francia.,556,570,1.021575e-06,West Francia.,556,570,1.021575e-06,King Charles III of West Francia,...,373,3.773069e-06,King Charles III of West Francia,341,373,5.532054e-09,King Charles III of West Francia,341,373,5.532054e-09
3,5ad39d53604f3c001a3fe8d4,"first half of the 10th century,",652,684,1.761726e-09,"first half of the 10th century,",652,684,4.405903e-01,first half of the 10th century,...,683,9.914104e-01,10th century,671,683,1.110426e-08,first half of the 10th century,653,683,5.637674e-01
4,5ad3a266604f3c001a3fea27,"political, cultural and military",30,63,1.606901e-07,"political, cultural and military",30,63,1.606901e-07,"political, cultural and military",...,63,1.533036e-10,"political, cultural and military",31,63,8.276052e-07,"political, cultural and military",31,63,8.276052e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5940,5ad28a57d7d075001a4299b3,nonconservative forces,187,210,1.125980e-08,nonconservative forces act to change the inte...,187,261,3.855142e-03,nonconservative forces,...,246,1.016702e-06,nonconservative forces and microscopic conserv...,35,155,3.846764e-12,nonconservative forces,188,210,4.124932e-02
5941,5ad28ad0d7d075001a4299cc,newton:,69,77,5.952701e-07,newton:,69,77,5.952701e-07,The pound-force,...,15,2.358760e-10,pound-force,4,15,1.539261e-10,pound-force,4,15,1.539261e-10
5942,5ad28ad0d7d075001a4299cd,kilogram-force,81,96,1.338633e-07,kilogram-force,81,96,1.338633e-07,kilogram-force,...,90,8.838547e-11,kilogram-force,82,96,2.608475e-08,kilogram-force,82,96,2.608475e-08
5943,5ad28ad0d7d075001a4299ce,kilogram-force,194,209,4.873671e-08,kilogram-force,194,209,4.873671e-08,kilogram-force,...,96,1.785797e-11,kilogram-force,195,209,9.138619e-10,kilogram-force,195,209,9.138619e-10


In [10]:
final_df.to_csv(f"{fname}.csv")